# LogBERT (HelenGuohx/logbert) on the BOLD exported splits


In [ ]:
%cd /content
!rm -rf logbert && git clone -q https://github.com/HelenGuohx/logbert.git
%cd /content/logbert
# compatibility: PyTorch >= 2.6 defaults torch.load(weights_only=True), which breaks loading the pickled model/center
!sed -i 's/torch.load(self.model_path)/torch.load(self.model_path, weights_only=False)/' bert_pytorch/predict_log.py
!sed -i 's/torch.load(self.model_dir + "best_center.pt")/torch.load(self.model_dir + "best_center.pt", weights_only=False)/' bert_pytorch/predict_log.py
!grep -n "weights_only" bert_pytorch/predict_log.py
# compatibility: NumPy >= 1.24 refuses ragged arrays; HDFS's variable-length sessions crash train/predict data assembly
!sed -i 's/logkey_seq_pairs = np.array(logkey_seq_pairs)/logkey_seq_pairs = np.array(logkey_seq_pairs, dtype=object)/' bert_pytorch/dataset/sample.py
!sed -i 's/time_seq_pairs = np.array(time_seq_pairs)/time_seq_pairs = np.array(time_seq_pairs, dtype=object)/' bert_pytorch/dataset/sample.py
!sed -i 's/log_seqs = np.array(log_seqs)/log_seqs = np.array(log_seqs, dtype=object)/' bert_pytorch/predict_log.py
!sed -i 's/tim_seqs = np.array(tim_seqs)/tim_seqs = np.array(tim_seqs, dtype=object)/' bert_pytorch/predict_log.py
!grep -n 'dtype=object' bert_pytorch/dataset/sample.py bert_pytorch/predict_log.py
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())

In [ ]:
# Mount Google Drive and unpack the inputs from My Drive/colab_logbert_official/
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile
DRIVE = '/content/drive/MyDrive/colab_logbert_official'
assert os.path.exists(f'{DRIVE}/logbert_inputs.zip'), f'logbert_inputs.zip not found in {DRIVE}'
if not os.path.exists('/content/inputs'):
    with zipfile.ZipFile(f'{DRIVE}/logbert_inputs.zip') as z: z.extractall('/content')
!ls /content/inputs && wc -l /content/inputs/*/*

In [ ]:
import os, re, json, shutil, subprocess, time, random

CONFIGS = ["bgl_chrono", "bgl_random", "hdfs_chrono", "hdfs_random"]   
MAX_TRAIN_HDFS = 100_000   # seeded subsample of benign HDFS training sessions
SEED = 0
RES = "/content/drive/MyDrive/colab_logbert_official/logbert_results"; os.makedirs(RES, exist_ok=True)  # persisted in Drive

def prepare(cfg):
    ds = "hdfs" if cfg.startswith("hdfs") else "bgl"
    out = f"/content/logbert/output/{cfg}/"; os.makedirs(out + "bert", exist_ok=True)
    src = f"/content/inputs/{cfg}/"
    lines = [l for l in open(src + "train").read().splitlines() if l.strip()]
    n_all = len(lines)
    if ds == "hdfs" and len(lines) > MAX_TRAIN_HDFS:
        random.Random(SEED).shuffle(lines); lines = lines[:MAX_TRAIN_HDFS]
    open(out + "train", "w").write("\n".join(lines) + "\n")
    for f in ["test_normal", "test_abnormal"]:
        shutil.copy(src + f, out + f)
    script = open(f"/content/logbert/{ds.upper()}/logbert.py").read()
    old = f'options["output_dir"] = "../output/{ds}/"'
    assert old in script, "official script layout changed"
    script = script.replace(old, f'options["output_dir"] = "../output/{cfg}/"')
    sdir = f"/content/logbert/{ds.upper()}"
    open(f"{sdir}/logbert_{cfg}.py", "w").write(script)
    drop = {}
    for f in ["test_normal", "test_abnormal"]:
        L = [len(l.split()) for l in open(src + f).read().splitlines() if l.strip()]
        drop[f] = {"n": len(L), "dropped_lt10": sum(1 for x in L if x < 10)}
    return sdir, {"train_sessions_used": len(lines), "train_sessions_available": n_all, "test_min_len_drop": drop}

def run(cfg):
    sdir, info = prepare(cfg)
    log = f"{RES}/{cfg}.log"; t0 = time.time()
    with open(log, "w") as fh:
        for mode in ["vocab", "train", "predict"]:
            fh.write(f"\n===== {mode} =====\n"); fh.flush()
            subprocess.run(["python", f"logbert_{cfg}.py", mode], cwd=sdir, stdout=fh, stderr=subprocess.STDOUT, check=False)
    txt = open(log).read()
    m = re.findall(r"best threshold: ([\d.]+), best threshold ratio: ([\d.]+)", txt)
    tp = re.findall(r"TP: (\d+), TN: (\d+), FP: (\d+), FN: (\d+)", txt)
    pr = re.findall(r"Precision: ([\d.]+)%, Recall: ([\d.]+)%, F1-measure: ([\d.]+)%", txt)
    info.update({"config": cfg, "wall_seconds": round(time.time() - t0),
                 "best_threshold_and_ratio": m[-1] if m else None,
                 "tp_tn_fp_fn": tp[-1] if tp else None,
                 "precision_recall_f1_pct": pr[-1] if pr else None,
                 "note": "official HelenGuohx/logbert code and defaults; threshold swept on the test set by its own predict step (test-tuned); sessions with <10 keys dropped by min_len=10"})
    json.dump(info, open(f"{RES}/{cfg}.json", "w"), indent=2)
    print(json.dumps(info, indent=2))
    return info

for cfg in CONFIGS:
    if os.path.exists(f"{RES}/{cfg}.json"):
        print("skip (done)", cfg); continue
    run(cfg)

In [ ]:
# Zip everything 
!cd /content/drive/MyDrive/colab_logbert_official && zip -qr logbert_official_results.zip logbert_results && ls -la
print("saved: My Drive/colab_logbert_official/logbert_official_results.zip")